# Fault tolerance

Stateless flows and Context read-only workloads are often easy to ensure fault-tolerance for because their idempotency is guaranteed, and failures can be detected and re-tried with few concerns.

The same is not true of a **stateful flow**. 

Consider the scenario in which we send, in one batch request a list of payloads for Rulebricks to solve against a stateful flow:

`[ {1}, {2}, {3} ]`

Where `{1}` directly determines the results of `{2}`, and `{2}` directly determines the result of `{3}`.

If `{1}` succeeds, but `{2}` fails, our execution system cannot continue with `{3}`, and any subsequent payloads that read/write to a particular context. Attempting to calculate `{3}` using the context state as of the end of `{1}` is wrong and may lead to an inaccurate result.

---

Below, we will look at generally addressing this problem. Simply, our Spark SDK helps you occasionally checkpoint Context states, such that you can resume from your last successful payload along with the Context state at that point. Treating Context state in this way restores idempotency and is the simplest way to develop resilience for long, fault-prone, stateful jobs. 

This solution may rightly create questions about whether the caller should simply store a state in memory and send it along with each payload, which is a viable approach for smaller workloads that do not require or meaningfully benefit from batch request performance (i.e. sending { a, b, ...state } is possible as a standalone request, but not possible in a single request shaped like [ { a1, b1 }, { a2, b2 }, ... { an, bn } ]). 

---

The following example processes account transactions. The SDK saves input, results, and verified Context state in your Delta storage. Recovery restores fresh physical Context keys and replays the uncheckpointed work.

## Setup

Use Spark 3.5 with Delta Lake and a storage location shared by every executor. Install `rulebricks-spark` on the driver and executors. Set the connection and published flow below; keep credentials in an environment variable or secret manager.

Start the Spark job with speculation disabled and `spark.task.maxFailures=1` and `spark.stage.maxConsecutiveAttempts=1`. Stateful HTTP retries are disabled by the SDK. One job must exclusively own these accounts and its checkpoint path.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, functions
from rulebricks.types import ContextSchema, ContextSchemaField

from rulebricks_spark import ExecutionError, contexts, resume

spark = (
    SparkSession.getActiveSession()
    or SparkSession.builder.appName("fault-tolerance")
    .config("spark.speculation", "false")
    .config("spark.task.maxFailures", "1")
    .config("spark.stage.maxConsecutiveAttempts", "1")
    .getOrCreate()
)
assert spark.sparkContext.getConf().get("spark.speculation", "false").lower() == "false"
assert spark.sparkContext.getConf().get("spark.task.maxFailures", "4") == "1"
assert spark.sparkContext.getConf().get("spark.stage.maxConsecutiveAttempts", "4") == "1"

connection = dict(
    api_key=os.environ["RULEBRICKS_API_KEY"],
    base_url=os.environ.get("RULEBRICKS_BASE_URL", "https://rulebricks.acme.com/api/v1"),
)
flow_slug = os.environ.get("RULEBRICKS_CHECKPOINT_FLOW_SLUG", "checkpointed-account-balance")
flow_version = os.environ.get("RULEBRICKS_CHECKPOINT_FLOW_VERSION", "1")
context_name = os.environ.get("RULEBRICKS_CHECKPOINT_CONTEXT_NAME", "Spark fault-tolerant balances")
output_root = os.environ.get("RULEBRICKS_CHECKPOINT_ROOT", "file:///tmp/rulebricks-examples")

## 1. Prepare a dedicated balance Context

Use a plain Context with no relationships, derived fields, history-dependent decisions, automatic bindings, or pending evaluations. Its complete decision state is `balance` and `last_sequence`. `ensure()` checks an existing definition without changing it.

Keep a stable business `account_id` separate from `context_id`, the physical instance used by this attempt. Recovery can change the physical key while preserving account and event identity.


In [ ]:
context_slug = contexts.ensure(
    context_name,
    schema=ContextSchema(
        base=[
            ContextSchemaField(key="id", name="Instance", type="string"),
            ContextSchemaField(key="balance", name="Balance", type="number", default_value=0),
            ContextSchemaField(
                key="last_sequence", name="Last event", type="number", default_value=0
            ),
        ]
    ),
    key="id",
    auto_execute_decisions=False,
    history_limit=1,
    ttl_seconds=604800,
    **connection,
)
print("Context for the published flow:", context_slug)

Publish the configured flow version with this contract before continuing:

| Step | Behavior |
| --- | --- |
| Input | Accept `account_id`, `context_id`, `event_id`, `sequence`, and `amount`. |
| Read | Read this Context using `context_id`. |
| Check order | Require `sequence == last_sequence + 1`; otherwise return an error without writing. |
| Update | Add `amount` to `balance` and write `last_sequence=sequence` in the same Context update. |
| Return | Return `account_id`, `event_id`, `balance`, and `last_sequence` under the flow’s `data` result key. |

Every Context operation, including in subflows, must use the supplied `context_id`. Pin the flow and its rule/subflow dependencies for the whole run. Return business identifiers, rather than physical Context keys, in the business result.


## 2. Seed fresh instances

This demonstration starts A and B at zero. Seeding is a Context mutation, so persist its receipts once and check them before execution. If seeding fails, stop and start a new attempt with fresh keys; do not blindly repeat it.

The checkpoint path below must be new. Record it outside the notebook so a replacement driver can recover the same run.


In [ ]:
run_id = str(uuid.uuid4())
checkpoint_path = f"{output_root}/{run_id}"
seed_path = f"{checkpoint_path}-seeds"
print("Keep this checkpoint path:", checkpoint_path)

events = spark.createDataFrame(
    [
        ("A", 1, "A-1", 100),
        ("A", 2, "A-2", -25),
        ("A", 3, "A-3", 50),
        ("B", 1, "B-1", 40),
        ("B", 2, "B-2", 20),
        ("B", 3, "B-3", -10),
    ],
    "account_id string, sequence long, event_id string, amount long",
).withColumn(
    "context_id", functions.concat(functions.lit(f"{run_id}:"), functions.col("account_id"))
)

initial = (
    events.select("context_id")
    .distinct()
    .withColumn("balance", functions.lit(0))
    .withColumn("last_sequence", functions.lit(0))
)
contexts.ingest(
    initial,
    context_slug,
    key="context_id",
    input_columns=["context_id", "balance", "last_sequence"],
    input_mapping={"context_id": "id"},
    **connection,
).write.format("delta").mode("errorifexists").save(seed_path)
seed_results = spark.read.format("delta").load(seed_path)
assert (
    seed_results.filter("rulebricks_meta.status IS NULL OR rulebricks_meta.status != 'ok'").count()
    == 0
)

## 3. Run with a checkpoint

`events_per_key=2` saves progress after two events per account in this small example. Choose a larger cadence for production to trade fewer checkpoint reads/writes against more replay work after failure. `max_batch_rows` controls request grouping within each checkpoint window.

With a checkpoint, `contexts.apply_flow()` executes and saves results before returning a materialized DataFrame. The SDK only advances the checkpoint after that window has completed and its state has been captured. An interrupted window stops the run; it is not silently skipped.


In [ ]:
checkpoint = contexts.Checkpoint(
    path=checkpoint_path,
    context=context_slug,
    context_key="context_id",
    events_per_key=2,
)

try:
    decisions = contexts.apply_flow(
        events,
        flow_slug,
        version=flow_version,
        key="account_id",
        order_by=["sequence"],
        checkpoint=checkpoint,
        max_batch_rows=512,
        input_columns=["account_id", "context_id", "event_id", "sequence", "amount"],
        correlation_id_path="event_id",
        execution_partitions=2,
        **connection,
    )
except ExecutionError as error:
    print("Run stopped. After service recovery, continue with the resume cell:", error)

## 4. Resume the saved run

For a real interruption, first stop the old producer and wait for the service to recover. Do not rerun the seeding or initial execution cells. In a new driver, run only the imports/session/connection setup, then put the saved checkpoint path in the cell below.

`resume(path)` loads the saved input and pinned execution contract. For an unfinished run it restores fresh Context keys from the last completed checkpoint and replays everything after that boundary, including events that may have committed in the abandoned attempt. Only one driver may resume a path at a time.

When the initial run completed normally, this same call simply reads its saved results without sending flow requests. This notebook does not manufacture an outage.


In [ ]:
checkpoint_to_resume = checkpoint_path  # On a new driver, replace with the saved path.
decisions = resume(checkpoint_to_resume, **connection)
decisions.select("account_id", "sequence", "rulebricks_meta.status").orderBy(
    "account_id", "sequence"
).show(truncate=False)

## Checks

All six events should succeed. The final balances are A=125 and B=50, with `last_sequence=3` for both. These checks read saved results and do not execute the flows again.

The completed checkpoint also exposes current physical Context IDs and saved state. This reads your Delta checkpoint; it does not call Rulebricks again.


In [ ]:
assert decisions.count() == 6
assert (
    decisions.filter("rulebricks_meta.status IS NULL OR rulebricks_meta.status != 'ok'").count()
    == 0
)
final_balances = decisions.filter("sequence = 3").select(
    "account_id",
    functions.get_json_object("rulebricks_result", "$.data.balance").cast("long").alias("balance"),
    functions.get_json_object("rulebricks_result", "$.data.last_sequence")
    .cast("long")
    .alias("last_sequence"),
)
expected = spark.createDataFrame(
    [("A", 125, 3), ("B", 50, 3)],
    "account_id string, balance long, last_sequence long",
)
assert final_balances.exceptAll(expected).count() == 0
assert expected.exceptAll(final_balances).count() == 0
final_balances.orderBy("account_id").show(truncate=False)

# Read the completed checkpoint, including actual IDs after a replay.
completed_state = contexts.Checkpoint(checkpoint_to_resume, context_name).state(spark)
completed_state.select("account_id", "context_id", "state").show(truncate=False)
checkpoint_balances = completed_state.select(
    "account_id",
    functions.get_json_object("state", "$.balance").cast("long").alias("balance"),
    functions.get_json_object("state", "$.last_sequence").cast("long").alias("last_sequence"),
)
assert checkpoint_balances.exceptAll(expected).count() == 0
assert expected.exceptAll(checkpoint_balances).count() == 0

## Next steps

Keep checkpoint Delta tables for as long as the job may need recovery. Use fresh checkpoint paths for independent jobs, and resume an existing path only through `resume(path)`. The physical `context_id` supplied in the original source remains in saved source columns; recovery manages replacement instance keys internally. Use the completed checkpoint’s `state(spark)` DataFrame to obtain the actual current physical IDs.

This example deliberately uses only current base fields. Relationships, tracked history, time-dependent decisions, automatic/pending evaluations, shared writers, or additional Context definitions require a recovery design that covers all of that state. A state JSON snapshot is not a general export of a Context's execution history.

Fresh keys isolate Context state. They do not undo external calls, emails, payments, decision logs, or usage from an abandoned attempt. Make those effects idempotent or reconcile them separately.
